In [0]:
# ============================================================
# GOLD LAYER — Aggregated Claims Metrics for Dashboards & ML
# ============================================================
from pyspark.sql.functions import *

CATALOG = "workspace"
SCHEMA  = "default"

df = spark.table(f"{CATALOG}.{SCHEMA}.claims_silver")

# --- Gold Table 1: Monthly Summary ---
df_monthly = df.groupBy("claim_year", "claim_month", "claim_type", "state") \
    .agg(
        count("claim_id").alias("total_claims"),
        countDistinct("policy_id").alias("unique_policies"),
        sum("damage_amount").alias("total_damage_amount"),
        sum("net_claim_amount").alias("total_net_payout"),
        avg("damage_amount").alias("avg_damage_amount"),
        sum(when(col("claim_status") == "Approved", 1).otherwise(0)).alias("approved_claims"),
        sum(when(col("claim_status") == "Denied",   1).otherwise(0)).alias("denied_claims"),
        sum(when(col("claim_status") == "Settled",  1).otherwise(0)).alias("settled_claims"),
        sum(when(col("is_suspicious_flag") == True, 1).otherwise(0)).alias("suspicious_claims"),
        sum(when(col("is_auto_approvable") == True, 1).otherwise(0)).alias("auto_approvable_claims"),
        avg("report_lag_days").alias("avg_report_lag_days"),
    ) \
    .withColumn("approval_rate",
        round(col("approved_claims") / col("total_claims") * 100, 2)) \
    .withColumn("fraud_rate",
        round(col("suspicious_claims") / col("total_claims") * 100, 2)) \
    .withColumn("auto_approval_rate",
        round(col("auto_approvable_claims") / col("total_claims") * 100, 2))

df_monthly.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.claims_gold_monthly")

# --- Gold Table 2: Fraud Scoring Features ---
df_fraud = df.select(
    "claim_id", "policy_id", "customer_id",
    "claim_type", "claim_status", "state",
    "damage_amount", "deductible", "net_claim_amount",
    "report_lag_days", "claim_severity",
    "vehicle_year", "vehicle_make",
    "is_suspicious_flag", "is_auto_approvable",
    "source_system"
).withColumn("fraud_score",
    # Rule-based fraud score (0–100); replace with ML model output later
    least(lit(100), greatest(lit(0),
        when(col("is_suspicious_flag") == True, 60).otherwise(0) +
        when(col("report_lag_days") > 2, 15).otherwise(0) +
        when(col("damage_amount") > 20000, 10).otherwise(0) +
        when(col("source_system") == "CallCenter", 5).otherwise(0) +
        when(col("claim_severity") == "Catastrophic", 10).otherwise(0)
    ))
).withColumn("fraud_risk_tier",
    when(col("fraud_score") >= 70, "High")
   .when(col("fraud_score") >= 40, "Medium")
   .otherwise("Low")
)

df_fraud.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.claims_gold_fraud_scores")

print("✅ Gold tables written:")
print(f"   → {CATALOG}.{SCHEMA}.claims_gold_monthly")
print(f"   → {CATALOG}.{SCHEMA}.claims_gold_fraud_scores")